# Music Files Summary Notebook
This notebook crawls a specified directory of music files, records file sizes and MP3 durations, saves the results to a CSV, and displays the measurements in a pandas DataFrame.

## 1. Import Required Libraries

In [ ]:
# Import libraries for filesystem operations and data handling
import os
from pathlib import Path
import pandas as pd
import dotenv

# Load environment variables from .env file
dotenv.load_dotenv()
LOCAL_DATA_PATH = os.getenv("LOCAL_DATA_PATH")
LOCAL_DEV_PATH = os.getenv("DEV_PATH")
base_path = Path(LOCAL_DATA_PATH)
# For reading mp3 metadata
from mutagen import File as MutagenFile

## 2. Define Directory Path

In [ ]:
# Directory to crawl
root_dir = Path(f"{LOCAL_DATA_PATH}/Sonica/audio/rehearsal archive")
# Base path to anonymize (can be changed later)
base_path = Path(LOCAL_DATA_PATH)
# Output location for results
output_dir = Path(f"{LOCAL_DEV_PATH}/rehersal-recording-processing")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
base_path
/home/sylvansky/dev/rehersal-recording-processing

## 3. Crawl Directory and Map Files with Size

In [ ]:
file_records = []
for dirpath, dirnames, filenames in os.walk(root_dir):
    for fname in filenames:
        path = Path(dirpath) / fname
        try:
            size = path.stat().st_size
        except Exception:
            size = None
        file_records.append({
            "filepath": str(path),
            "size_bytes": size,
            "duration_seconds": None,
        })

len(file_records), file_records[:5]

## 4. Assess MP3 Durations

In [ ]:
for rec in file_records:
    if rec["filepath"].lower().endswith(".mp3") and rec["size_bytes"] is not None:
        try:
            audio = MutagenFile(rec["filepath"])
            if audio is not None and hasattr(audio, "info"):
                rec["duration_seconds"] = audio.info.length
        except Exception:
            rec["duration_seconds"] = None

file_records[:5]

## 5. Create DataFrame from Results

In [ ]:
df = pd.DataFrame(file_records)
if not df.empty:
    df["file_extension"] = df["filepath"].apply(lambda p: Path(p).suffix.lower().lstrip("."))
    # Anonymize file paths by removing the base path
    df["filepath_anonymized"] = df["filepath"].apply(
        lambda p: str(Path(p).relative_to(base_path))
    )
    #drop the original filepath column to avoid confusion
    df.drop(columns=["filepath"], inplace=True)
    # Optionally convert sizes to MB and durations to minutes for readability
    df["size_mb"] = df["size_bytes"] / (1024 * 1024)
    df["duration_min"] = df["duration_seconds"] / 60

df.head()

## 6. Display DataFrame

In [ ]:
df.file_extension.value_counts()

In [ ]:
df_filtered = df[df["file_extension"].isin(["mp3", "wav"])].copy()

In [ ]:
df_filtered.duration_min.sum()/60

In [ ]:
df_filtered.size_mb.sum()

In [ ]:
df_filtered.count()

## 7. Save Results to CSV

In [ ]:
csv_path = output_dir / "files_to_process.csv"
df_filtered.to_csv(csv_path, index=False)
print(f"Saved summary to {csv_path}")